In [3]:
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from PIL import Image
import requests

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
except userdata.SecretNotFoundError:
    print("Error: HF_TOKEN not found in Secrets. Please add it via the key icon on the left.")

class NLPModule:
    """
    A unified OOP interface for 10 HuggingFace NLP/ML tasks.
    Models are loaded lazily — only instantiated on first use.
    """

    def __init__(self):
        # Private pipeline holders — None until first use
        self._sentiment_pipe     = None
        self._zero_shot_pipe     = None
        self._text_gen_pipe      = None
        self._mask_fill_pipe     = None
        self._ner_pipe           = None
        self._qa_pipe            = None
        self._summarizer_tokenizer = None # Changed for summarization
        self._summarizer_model = None    # Changed for summarization
        self._translator_tokenizer = None # Changed for translation
        self._translator_model = None     # Changed for translation
        self._image_class_pipe   = None
        self._asr_pipe           = None

    # ──────────────────────────────────────────────
    # Internal lazy-loaders
    # ──────────────────────────────────────────────
    def _get_sentiment(self):
        if self._sentiment_pipe is None:
            try:
                self._sentiment_pipe = pipeline("sentiment-analysis")
            except Exception as e:
                raise RuntimeError(f"Error loading sentiment-analysis pipeline: {e}")
        return self._sentiment_pipe

    def _get_zero_shot(self):
        if self._zero_shot_pipe is None:
            try:
                self._zero_shot_pipe = pipeline("zero-shot-classification")
            except Exception as e:
                raise RuntimeError(f"Error loading zero-shot-classification pipeline: {e}")
        return self._zero_shot_pipe

    def _get_text_gen(self):
        if self._text_gen_pipe is None:
            try:
                self._text_gen_pipe = pipeline("text-generation", model="gpt2")
            except Exception as e:
                raise RuntimeError(f"Error loading text-generation pipeline: {e}")
        return self._text_gen_pipe

    def _get_mask_fill(self):
        if self._mask_fill_pipe is None:
            try:
                self._mask_fill_pipe = pipeline("fill-mask", model="bert-base-uncased")
            except Exception as e:
                raise RuntimeError(f"Error loading fill-mask pipeline: {e}")
        return self._mask_fill_pipe

    def _get_ner(self):
        if self._ner_pipe is None:
            try:
                self._ner_pipe = pipeline("ner", aggregation_strategy="simple")
            except Exception as e:
                raise RuntimeError(f"Error loading NER pipeline: {e}")
        return self._ner_pipe

    def _get_qa(self):
        if self._qa_pipe is None:
            try:
                self._qa_pipe = pipeline("question-answering")
            except Exception as e:
                raise RuntimeError(f"Error loading question-answering pipeline: {e}")
        return self._qa_pipe

    def _get_summarizer_components(self):
        if self._summarizer_model is None or self._summarizer_tokenizer is None:
            try:
                model_name = "facebook/bart-large-cnn"
                self._summarizer_tokenizer = AutoTokenizer.from_pretrained(model_name)
                self._summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
            except Exception as e:
                raise RuntimeError(f"Error loading summarization model/tokenizer: {e}")
        return self._summarizer_tokenizer, self._summarizer_model

    def _get_translator_components(self):
        if self._translator_model is None or self._translator_tokenizer is None:
            try:
                model_name = "Helsinki-NLP/opus-mt-tc-big-en-tr"
                self._translator_tokenizer = AutoTokenizer.from_pretrained(model_name)
                self._translator_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
            except Exception as e:
                raise RuntimeError(f"Error loading translation model/tokenizer: {e}")
        return self._translator_tokenizer, self._translator_model

    def _get_image_classifier(self):
        if self._image_class_pipe is None:
            try:
                self._image_class_pipe = pipeline(
                    "image-classification",
                    model="google/vit-base-patch16-224"
                )
            except Exception as e:
                raise RuntimeError(f"Error loading image-classification pipeline: {e}")
        return self._image_class_pipe

    def _get_asr(self):
        if self._asr_pipe is None:
            try:
                self._asr_pipe = pipeline(
                    "automatic-speech-recognition",
                    model="openai/whisper-medium",
                    chunk_length_s=30
                )
            except Exception as e:
                raise RuntimeError(f"Error loading automatic-speech-recognition pipeline: {e}")
        return self._asr_pipe

    # ──────────────────────────────────────────────
    # Public API methods
    # ──────────────────────────────────────────────

    def sentiment_analysis(self, text: str) -> str:
        """
        Returns POSITIVE or NEGATIVE sentiment with confidence score.
        Accepts a single string or newline-separated strings.
        """
        sentences = [s.strip() for s in text.strip().split("\n") if s.strip()]
        results = self._get_sentiment()(sentences)
        output = ""
        for s, r in zip(sentences, results):
            output += f"Text   : {s}\nResult : {r['label']} (score: {r['score']:.4f})\n\n"
        return output.strip()

    def zero_shot_classification(self, text: str, labels: str) -> str:
        """
        Classifies text into user-provided comma-separated candidate labels.
        """
        candidate_labels = [l.strip() for l in labels.split(",") if l.strip()]
        result = self._get_zero_shot()(text, candidate_labels=candidate_labels)
        output = f"Text: {text}\n\nScores:\n"
        for label, score in zip(result["labels"], result["scores"]):
            output += f"  {label:<20}: {score:.4f}\n"
        return output

    def text_generation(self, prompt: str, max_length: int = 60, num_sequences: int = 2) -> str:
        """
        Completes an incomplete sentence. Returns `num_sequences` alternatives.
        """
        results = self._get_text_gen()(
            prompt,
            max_length=max_length,
            num_return_sequences=num_sequences,
            truncation=True
        )
        output = ""
        for i, r in enumerate(results, 1):
            output += f"Option {i}:\n{r['generated_text']}\n\n"
        return output.strip()

    def mask_filling(self, sentence: str) -> str:
        """
        Fills [MASK] token in a BERT-style sentence.
        Use [MASK] as the placeholder in your sentence.
        """
        results = self._get_mask_fill()(sentence)
        output = f"Original: {sentence}\n\nTop predictions:\n"
        for r in results[:5]:
            output += f"  → '{r['token_str']}' (score: {r['score']:.4f})\n    {r['sequence']}\n\n"
        return output.strip()

    def named_entity_recognition(self, text: str) -> str:
        """
        Extracts named entities (PER, ORG, LOC, MISC) from the given text.
        """
        entities = self._get_ner()(text)
        output = f"Text: {text}\n\nEntities found:\n"
        for e in entities:
            output += f"  [{e['entity_group']:<6}] {e['word']}  (score: {e['score']:.4f})\n"
        return output

    def question_answering(self, question: str, context: str) -> str:
        """
        Extracts the answer to `question` from the provided `context` paragraph.
        """
        result = self._get_qa()(question=question, context=context)
        return (
            f"Question : {question}\n"
            f"Answer   : {result['answer']}\n"
            f"Score    : {result['score']:.4f}\n"
            f"Position : chars {result['start']}–{result['end']}"
        )

    def summarization(self, text: str, max_length: int = 80, min_length: int = 30) -> str:
        """
        Produces an abstractive summary of a longer passage.
        """
        try:
            tokenizer, model = self._get_summarizer_components()
            inputs = tokenizer([text], max_length=1024, return_tensors='pt', truncation=True)
            summary_ids = model.generate(
                inputs['input_ids'],
                num_beams=4,
                max_length=max_length,
                min_length=min_length,
                early_stopping=True
            )
            summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            return f"Summary:\n{summary_text}"
        except RuntimeError as e:
            # Catch errors from lazy loader
            return f"Error during summarization: {e}"
        except Exception as e:
            # Catch any other unexpected errors during summarization inference
            return f"Error during summarization inference: {e}"

    def translation(self, text: str) -> str:
        """
        Translates English text to Turkish using Helsinki-NLP opus-mt.
        """
        try:
            tokenizer, model = self._get_translator_components()
            inputs = tokenizer(text, return_tensors="pt")
            translated_tokens = model.generate(**inputs)
            translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
            return f"EN: {text}\n\nTR: {translated_text}"
        except RuntimeError as e:
            # Catch errors from lazy loader
            return f"Error during translation: {e}"
        except Exception as e:
            # Catch any other unexpected errors during translation inference
            return f"Error during translation inference: {e}"

    def image_classification(self, image_input) -> str:
        """
        Classifies an image using Google ViT.
        `image_input` can be a PIL Image, a file path, or a URL string.
        """
        if isinstance(image_input, str) and image_input.startswith("http"):
            image_input = Image.open(requests.get(image_input, stream=True).raw)

        predictions = self._get_image_classifier()(image_input)
        output = "Top predictions:\n"
        for p in predictions[:5]:
            output += f"  {p['label']:<35} score: {p['score']:.4f}\n"
        return output

    def speech_recognition(self, audio_input) -> str:
        """
        Transcribes audio using OpenAI Whisper.
        `audio_input` can be a file path, URL, or numpy array.
        """
        result = self._get_asr()(audio_input)
        return f"Transcription:\n{result['text']}"


Successfully logged into Hugging Face Hub!


In [16]:
pip install --upgrade transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 33.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [4]:
import gradio as gr

# Single shared instance — models are loaded lazily per tab
nlp = NLPModule()

# ─────────────────────────────────────────────────────────
# Wrapper functions (thin adapters between Gradio & module)
# ─────────────────────────────────────────────────────────

def run_sentiment(text):
    return nlp.sentiment_analysis(text)

def run_zero_shot(text, labels):
    return nlp.zero_shot_classification(text, labels)

def run_text_gen(prompt, max_len, num_seq):
    return nlp.text_generation(prompt, max_length=int(max_len), num_sequences=int(num_seq))

def run_mask_fill(sentence):
    return nlp.mask_filling(sentence)

def run_ner(text):
    return nlp.named_entity_recognition(text)

def run_qa(question, context):
    return nlp.question_answering(question, context)

def run_summarization(text):
    return nlp.summarization(text)

def run_translation(text):
    return nlp.translation(text)

def run_image_classification(image):
    return nlp.image_classification(image)

def run_asr(audio_path):
    return nlp.speech_recognition(audio_path)


# ─────────────────────────────────────────────────────────
# Gradio UI — one tab per task
# ─────────────────────────────────────────────────────────

with gr.Blocks(title="EE471 Week 10 — NLP Demo", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# EE471 Week 10 — HuggingFace NLP Tasks\nInteractive demo for all 10 pipeline tasks.")

    # ── Tab 1: Sentiment Analysis ──────────────────────
    with gr.Tab("1 · Sentiment Analysis"):
        gr.Markdown("Enter one sentence per line. Each will be classified as POSITIVE or NEGATIVE.")
        with gr.Row():
            sa_input  = gr.Textbox(label="Input sentences (one per line)",
                                   value="I've been waiting for a EE471 course my whole life.\nI hate EE471 course",
                                   lines=4)
            sa_output = gr.Textbox(label="Results", lines=6)
        gr.Button("Analyse").click(run_sentiment, inputs=sa_input, outputs=sa_output)

    # ── Tab 2: Zero-Shot Classification ───────────────
    with gr.Tab("2 · Zero-Shot Classification"):
        gr.Markdown("Classify text into any labels you provide (no training needed).")
        zs_text   = gr.Textbox(label="Text",
                               value="Berkshire keeps their cash reserves at an extremely high level.")
        zs_labels = gr.Textbox(label="Candidate labels (comma-separated)",
                               value="finance, politics, technology, sports, economics")
        zs_output = gr.Textbox(label="Scores", lines=8)
        gr.Button("Classify").click(run_zero_shot, inputs=[zs_text, zs_labels], outputs=zs_output)

    # ── Tab 3: Text Generation ─────────────────────────
    with gr.Tab("3 · Text Generation"):
        gr.Markdown("Provide an incomplete sentence — the model will complete it.")
        tg_prompt  = gr.Textbox(label="Prompt",
                                value="If I continue to successfully complete all in-class exercises in EE471 course,")
        with gr.Row():
            tg_maxlen = gr.Slider(20, 100, value=35, step=5, label="Max length (words)")
            tg_num    = gr.Slider(1, 4, value=2, step=1, label="# of alternatives")
        tg_output  = gr.Textbox(label="Generated continuations", lines=8)
        gr.Button("Generate").click(run_text_gen, inputs=[tg_prompt, tg_maxlen, tg_num], outputs=tg_output)

    # ── Tab 4: Mask Filling ────────────────────────────
    with gr.Tab("4 · Mask Filling"):
        gr.Markdown("Place `[MASK]` where you want the model to fill in a word.")
        mf_input  = gr.Textbox(label="Masked sentence",
                               value="To understand generative AI, one must study [MASK] well.")
        mf_output = gr.Textbox(label="Top predictions", lines=10)
        gr.Button("Fill Mask").click(run_mask_fill, inputs=mf_input, outputs=mf_output)

    # ── Tab 5: Named Entity Recognition ───────────────
    with gr.Tab("5 · NER"):
        gr.Markdown("Extracts **persons**, **organizations**, and **locations** from text.")
        ner_input  = gr.Textbox(label="Text",
                                value="I am Nate, a research assistant in Izmir Institute of Technology, "
                                      "and currently living and working in beautiful city İzmir in Türkiye.",
                                lines=3)
        ner_output = gr.Textbox(label="Entities", lines=8)
        gr.Button("Extract Entities").click(run_ner, inputs=ner_input, outputs=ner_output)

    # ── Tab 6: Question Answering ──────────────────────
    with gr.Tab("6 · Question Answering"):
        gr.Markdown("Answer a question based on a given context paragraph.")
        qa_question = gr.Textbox(label="Question", value="What is the name of the person?")
        qa_context  = gr.Textbox(label="Context",
                                 value="I am Nate, a research assistant in Izmir Institute of Technology, "
                                       "and currently living and working in beautiful city İzmir in Türkiye.",
                                 lines=4)
        qa_output   = gr.Textbox(label="Answer", lines=5)
        gr.Button("Answer").click(run_qa, inputs=[qa_question, qa_context], outputs=qa_output)

    # ── Tab 7: Summarization ───────────────────────────
    with gr.Tab("7 · Summarization"):
        gr.Markdown("Paste a long passage and get a concise summary.")
        sm_input  = gr.Textbox(label="Long text", lines=8,
                               value=(
                                   "The 2008 Global Financial Crisis stands as the most severe economic collapse "
                                   "of the 21st century, often compared to the Great Depression of the 1930s. "
                                   "Triggered by the bursting of the United States housing bubble, its effects "
                                   "rippled across the globe, leading to the collapse of major financial institutions "
                                   "and a deep international recession. The crisis began with the subprime mortgage "
                                   "market. In the early 2000s, low interest rates and a push for homeownership led "
                                   "banks to issue high-risk loans to borrowers with poor credit."
                               ))
        sm_output = gr.Textbox(label="Summary", lines=4)
        gr.Button("Summarize").click(run_summarization, inputs=sm_input, outputs=sm_output)

    # ── Tab 8: Translation ─────────────────────────────
    with gr.Tab("8 · Translation (EN → TR)"):
        gr.Markdown("Translate English text to Turkish using Helsinki-NLP.")
        tr_input  = gr.Textbox(label="English text",
                               value="The 2008 Global Financial Crisis stands as the most severe economic collapse "
                                     "of the 21st century, often compared to the Great Depression.")
        tr_output = gr.Textbox(label="Turkish translation", lines=4)
        gr.Button("Translate").click(run_translation, inputs=tr_input, outputs=tr_output)

    # ── Tab 9: Image Classification ───────────────────
    with gr.Tab("9 · Image Classification"):
        gr.Markdown("Upload any image — Google ViT will predict its class.")
        ic_input  = gr.Image(type="pil", label="Upload image")
        ic_output = gr.Textbox(label="Top predictions", lines=7)
        gr.Button("Classify Image").click(run_image_classification, inputs=ic_input, outputs=ic_output)

    # ── Tab 10: Speech Recognition ─────────────────────
    with gr.Tab("10 · Speech Recognition"):
        gr.Markdown("Record or upload an audio clip — Whisper will transcribe it.")
        asr_input  = gr.Audio(type="filepath", label="Audio input")
        asr_output = gr.Textbox(label="Transcription", lines=4)
        gr.Button("Transcribe").click(run_asr, inputs=asr_input, outputs=asr_output)


demo.launch(share=True)   # share=True gives a public HuggingFace URL for demo

/tmp/ipykernel_51544/635722423.py:45: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="EE471 Week 10 — NLP Demo", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1195a3e76fc4fee055.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
